In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_config

In [0]:
%run /Workspace/Users/bernabeglennmarkc@gmail.com/superstore/utilities/util_helpers

### 1. Read from Bronze

In [0]:
log("Reading from Bronze ...")
df_bronze = spark.table(TBL_BRONZE_RAW)

In [0]:
display(df_bronze.limit(10))

### 2. Extract Order Columns & Add Ingestion Metadata

In [0]:
df_orders = df_bronze.select(
    F.col("row_id"),
    F.col("order_id"),
    F.col("order_date"),
    F.col("ship_date"),
    F.col("ship_mode"),
    F.col("customer_id"),
    F.col("product_id"),
    F.col("country"),
    F.col("city"),
    F.col("state"),
    F.col("postal_code"),
    F.col("region"),
    F.col("sales"),
    F.col("quantity"),
    F.col("discount"),
    F.col("profit"),
    F.col("ingested_at"),
    F.col("file_path"),
    F.col("file_name"),
    F.col("file_size")
) \
.withColumn("transformed_at", F.current_timestamp()) 
display(df_orders.limit(10))

In [0]:
df_orders.printSchema()

### 3. Change Data Type 

In [0]:
# Check what's inside the numeric columns
df_orders.select("sales", "quantity", "discount", "profit").show(20, truncate=False)

# Check for non-numeric values
df_orders.filter(~F.col("sales").rlike(r"^\d+(\.\d+)?$")).select("sales").show(20, truncate=False)

In [0]:
df_orders = df_orders \
    .withColumn("order_date", F.try_to_date(F.col("order_date"), "M/d/yyyy")) \
    .withColumn("ship_date", F.try_to_date(F.col("ship_date"), "M/d/yyyy")) \
    .withColumn("sales", F.col("sales").cast("double")) \
    .withColumn("quantity", F.col("quantity").cast("int")) \
    .withColumn("discount", F.col("discount").cast("double")) \
    .withColumn("profit", F.col("profit").cast("double"))

display(df_orders.limit(10))
df_orders.printSchema()